Лабораторна робота 2. Харченко, ФІТ 3-10

---

# Завдання 1

Обробка та нааліз даних про ВВП країн. Датасет було отримано з Вікіпедії з використанням бібілотеки Pandas.

1. Вивести перші 5 рядків датасета

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from io import StringIO

url = "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)"
# без User-Agent вікіпедія не віддає сторінку (помилка 403)
html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text

# беремо таблицю, де є стовпець Country/Territory
df = pd.read_html(StringIO(html), match="Country/Territory")[0]

In [ ]:
df.head()

2.	Визначити розмір датасета.

In [ ]:
print("Рядків:", df.shape[0])
print("Стовпців:", df.shape[1])

 3.	Визначити оптимальну кількість стовпців (видалити зайві, на власний розсуд).

In [ ]:
print(df.columns.tolist())

# в таблиці 4 стовпці: країна і три оцінки ВВП (млн доларів), всі вони потрібні
# тому прибираємо тільки виноски [1], [n 1] з назв
df.columns = ["Country", "IMF (2026)", "World Bank (2025)", "United Nations (2024)"]
df["Country"] = df["Country"].str.replace(r"\[.*?\]", "", regex=True)

# рядок World - це сума по всіх країнах, а не країна, він тільки зіпсує статистику
df = df[df["Country"] != "World"].reset_index(drop=True)
df.head()

4.	Замініть у таблиці значення "—" на значення NaN. Перевірити наявність пропущених значень. При наявності, замінити пропущені значення на середнє значення.

In [ ]:
cols = ["IMF (2026)", "World Bank (2025)", "United Nations (2024)"]

# зараз пропуски в таблиці записані як "—N/a", тому замінюємо все, що починається з "—"
df[cols] = df[cols].replace(r"^—.*", np.nan, regex=True)

In [ ]:
# в деяких числах є коми і рік в дужках, наприклад "1,982 (2019)"
# прибираємо це і переводимо стовпці в числа
for col in cols:
    df[col] = df[col].str.replace(r"\(\d{4}\)", "", regex=True).str.replace(",", "").str.strip()
    df[col] = pd.to_numeric(df[col])

df.dtypes

In [ ]:
df.isna().sum()

In [ ]:
# запам'ятовуємо, в яких країнах були пропуски, знадобиться далі
had_nan = df[cols].isna().any(axis=1)
print("Країни з пропусками:", df.loc[had_nan, "Country"].tolist())

# заповнюємо пропуски середнім по стовпцю
df[cols] = df[cols].fillna(df[cols].mean())

In [ ]:
df.isna().sum()

5. Ще раз вивести датасет

In [ ]:
df

6.	Перевірити наявність дублікатів. При наявності видалити дублікати.

In [ ]:
print("Однакових рядків:", df.duplicated().sum())

In [ ]:
print("Країн, що повторюються:", df["Country"].duplicated().sum())

In [ ]:
df = df.drop_duplicates()
print("Розмір після видалення дублікатів:", df.shape)

7.	Вивести описову статистику датасету describe()

In [ ]:
df.describe()

8. Визначте відхилення (різницю) між показниками IMF (2026) та World Bank (2025) для кожної країни. У яких країнах ці показники найбільше відрізняються (дати відповідь)?

In [ ]:
diff = df[["Country", "IMF (2026)", "World Bank (2025)"]].copy()
diff["Різниця"] = diff["IMF (2026)"] - diff["World Bank (2025)"]
diff["Різниця по модулю"] = diff["Різниця"].abs()

print("Топ-10 за різницею (всі країни):")
display(diff.sort_values("Різниця по модулю", ascending=False).head(10))

# у країн з пропусками одне значення - це середнє, тому різниця там штучна
print("Топ-10 тільки по країнах без пропусків:")
display(diff[~had_nan].sort_values("Різниця по модулю", ascending=False).head(10))

**Відповідь:** найбільше показники IMF і World Bank відрізняються у США (приблизно на 1,6 трлн доларів) і в Китаї (приблизно на 1,35 трлн). Далі йдуть Німеччина, Бразилія і Австралія.

У першій таблиці в топ потрапили маленькі острови (Saint Martin, American Samoa і т.д.), але це тільки через те, що в них пропуск заповнили середнім значенням, тобто це не справжня різниця.

9.	Обчисліть кореляцію між показниками IMF (2026), World Bank (2025) та United Nations (2024). Які пари змінних мають найвищу кореляцію?

In [ ]:
corr = df[cols].corr()
corr

In [ ]:
print("IMF і World Bank:", round(corr.loc["IMF (2026)", "World Bank (2025)"], 4))
print("IMF і United Nations:", round(corr.loc["IMF (2026)", "United Nations (2024)"], 4))
print("World Bank і United Nations:", round(corr.loc["World Bank (2025)", "United Nations (2024)"], 4))

**Відповідь:** всі три показники дуже сильно корелюють між собою, коефіцієнти близько 0.998. Найвища кореляція між World Bank (2025) і United Nations (2024), 0.9984. Тобто всі три джерела оцінюють ВВП країн майже однаково.

Обчислити середнє значення за стовпцями

In [ ]:
df[cols].mean()

In [ ]:
# для порівняння медіана
df[cols].median()

In [ ]:
print("У скільки разів середнє більше за медіану:")
print((df[cols].mean() / df[cols].median()).round(1))

**Висновок:** середнє значення в рази більше за медіану, бо кілька дуже великих економік (США, Китай) сильно тягнуть середнє вгору, а в більшості країн ВВП набагато менший.

11.	Обчисліть стандартне відхилення показників для кожної країни. Яка країна має найвищу варіативність у показниках між роками?

In [ ]:
# стандартне відхилення трьох показників для кожної країни (по рядку)
std = df[["Country"]].copy()
std["Стандартне відхилення"] = df[cols].std(axis=1)

print("Топ-10 (всі країни):")
display(std.sort_values("Стандартне відхилення", ascending=False).head(10))

# тут теж в топ лізуть країни з пропусками, бо в них одне значення - це середнє
print("Топ-10 тільки по країнах без пропусків:")
display(std[~had_nan].sort_values("Стандартне відхилення", ascending=False).head(10))

**Відповідь:** найбільша варіативність у США (стандартне відхилення приблизно 1,5 трлн доларів), далі Китай і Німеччина. Це логічно, бо у великих економік навіть невелика зміна у відсотках дає велику різницю в доларах.

Острови в першій таблиці знову через заповнення пропусків середнім.

12.	Визначення країни з найвищим та найнижчим показниками: Знайдіть країну з найвищим та найнижчим показниками у кожному з років (IMF (2026), World Bank (2025), United Nations (2024)).

In [ ]:
for col in cols:
    max_country = df.loc[df[col].idxmax(), "Country"]
    min_country = df.loc[df[col].idxmin(), "Country"]
    print(f"{col}: найвищий - {max_country} ({df[col].max():.0f}), найнижчий - {min_country} ({df[col].min():.0f})")

**Відповідь:** в усіх трьох роках найвищий ВВП у США, а найнижчий у Тувалу.

13.	Побудуйте гістограму для розподілу показників IMF (2026) серед всіх країн. Який вигляд має розподіл? Чи є країни, що виділяються?

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df["IMF (2026)"], bins=40)
plt.title("Розподіл ВВП країн, IMF (2026)")
plt.xlabel("ВВП, млн доларів")
plt.ylabel("Кількість країн")
plt.show()

print("Асиметрія:", round(df["IMF (2026)"].skew(), 2))
print(df.nlargest(5, "IMF (2026)")[["Country", "IMF (2026)"]])

**Відповідь:** розподіл дуже несиметричний, скошений вправо (асиметрія майже 10). Більшість країн мають порівняно невеликий ВВП і потрапляють у перший стовпчик гістограми. Сильно виділяються США і Китай, менше Німеччина, Японія і Велика Британія.

14.	Розрахуйте частку кожної країни в загальному значенні для кожного року (IMF (2026), World Bank (2025), United Nations (2024)).
 Як змінюються частки країн з часом (дати відповідь)?

In [ ]:
# значення, заповнені середнім, сильно завищують загальну суму,
# тому частки рахуємо по країнах без пропусків
real = df[~had_nan]

# частка країни = її ВВП / сума ВВП всіх країн * 100
shares = real[cols] / real[cols].sum() * 100
shares.insert(0, "Country", real["Country"])
shares["Зміна з 2024 по 2026"] = shares["IMF (2026)"] - shares["United Nations (2024)"]

print("Частки 10 найбільших економік, %:")
display(shares.sort_values("IMF (2026)", ascending=False).head(10).round(2))

print("Найбільше зросла частка:")
display(shares.sort_values("Зміна з 2024 по 2026", ascending=False).head(5).round(3))
print("Найбільше впала частка:")
display(shares.sort_values("Зміна з 2024 по 2026").head(5).round(3))

**Відповідь:** частки змінюються мало, в основному на десяті частки відсотка. Найбільші частки у США (близько 26%) і Китаю (близько 17%), і саме в них частка трохи зменшується з 2024 до 2026 (у США з 26,5% до 25,8%). Найбільше частка виросла у Росії, Німеччини і Бразилії.

Частки рахували тільки по країнах без пропусків, бо значення, заповнені середнім, завищують загальну суму.

15.	Візуалізуйте зміни в показниках для кожної країни за три роки на графіку. Які країни показують стабільне зростання або спад (дати відповідь)?

In [ ]:
years = ["United Nations (2024)", "World Bank (2025)", "IMF (2026)"]

# всі 200+ країн на одному графіку не прочитати, тому беремо 10 найбільших
top10 = df.nlargest(10, "IMF (2026)").set_index("Country")[years]
top10.columns = ["2024", "2025", "2026"]
top10.T.plot(marker="o", figsize=(10, 6))
plt.title("ВВП 10 найбільших економік за три роки")
plt.ylabel("ВВП, млн доларів")
plt.yscale("log")  # логарифмічна шкала, бо США і Китай набагато більші за інших
plt.legend(bbox_to_anchor=(1, 1))
plt.show()

# для всіх країн (без тих, де були пропуски) рахуємо, хто росте або падає всі три роки
real = df[~had_nan]
growth = (real[years[0]] < real[years[1]]) & (real[years[1]] < real[years[2]])
fall = (real[years[0]] > real[years[1]]) & (real[years[1]] > real[years[2]])
print("Країн всього:", len(real))
print("Стабільно ростуть:", growth.sum())
print("Стабільно падають:", fall.sum(), real.loc[fall, "Country"].tolist())
print("Без чіткої тенденції:", len(real) - growth.sum() - fall.sum())

**Відповідь:** з 10 найбільших економік стабільно ростуть 9, тільки Японія у 2025 виросла, а у 2026 трохи впала. Серед усіх країн без пропусків стабільно ростуть 159 з 196, стабільно падають тільки Іран і Ефіопія, у решти 35 немає чіткої тенденції.

Треба враховувати, що три стовпці взяті з різних джерел (ООН, Світовий банк, МВФ), тому це не зовсім один і той самий показник по роках.